# Module 3 – Motifs and Discords
**Dataset**: CMI Wrist Accelerometer — loaded from Module 0 outputs  
**Channels**: `enmo` (activity intensity), `anglez` (posture/orientation)  
**Target**: `sii_binary` (0 = non-problematic, 1 = problematic)

**Pipeline**:
1. Load Module 0 outputs (no re-cleaning)
2. Z-score normalize per TS per channel
3. **Data-driven window length selection** (separation score, m=10–60)
4. Representative TS selection (systematic)
5. Motif discovery — single TS with exclusion zone
6. Discord discovery — single TS with exclusion zone
7. Motif vs discord comparison
8. Global per-subject matrix profile (no concatenation)
9. **Consensus motif by class ± std shading**
10. Motif regularity score by class + Mann-Whitney
11. Class-level discord analysis + Mann-Whitney
12. **Fisher exact test — discord enrichment**
13. **MP distribution by class**
14. Discord density over time
15. Cross-series motif + discord matching
16. **Shapelet bridge — consensus motif as primitive shapelet**
17. Joint enmo + anglez discord
18. Repeat for anglez
19. Shapelet alignment placeholder (after classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import stumpy
from scipy import stats as sp_stats
from scipy.stats import fisher_exact
from scipy.spatial.distance import euclidean
from numpy.linalg import norm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SIGNAL_COLS = ['X','Y','Z','enmo','anglez','light','battery_voltage']
ENMO_IDX   = SIGNAL_COLS.index('enmo')
ANGLEZ_IDX = SIGNAL_COLS.index('anglez')
print('Setup complete.')

## Step 1 – Load Module 0 Outputs

In [ ]:
X_train_raw = np.load('X_train_raw.npy').astype(np.float64)  # (N, 200, 7)
y_train     = np.load('y_train.npy')
meta_train  = pd.read_csv('meta_train.csv')

assert not np.isnan(X_train_raw).any(), 'NaNs found — check Module 0 outputs'

N = len(X_train_raw)
print(f'X_train_raw : {X_train_raw.shape}  (N x 200 timesteps x 7 channels)')
print(f'y_train     : class 0={( y_train==0).sum()}, class 1={(y_train==1).sum()}')
print(f'Source      : Module 0 clean outputs — no re-cleaning applied')

## Step 2 – Z-Score Normalization (Per TS Per Channel)

In [ ]:
def zscore(arr):
    """Z-score a 1D array. Returns float64."""
    arr = arr.astype(np.float64)
    std = arr.std()
    return (arr - arr.mean()) / (std + 1e-8)

def zscore_all(signals_array, ch_idx):
    """Z-score one channel across all TS. Returns (N, 200)."""
    ch = signals_array[:, :, ch_idx].copy().astype(np.float64)
    mu  = ch.mean(axis=1, keepdims=True)
    std = ch.std(axis=1, keepdims=True) + 1e-8
    return (ch - mu) / std

enmo_z_all   = zscore_all(X_train_raw, ENMO_IDX)    # (N, 200)
anglez_z_all = zscore_all(X_train_raw, ANGLEZ_IDX)  # (N, 200)

print(f'enmo_z_all   : {enmo_z_all.shape}  per-TS mean≈{enmo_z_all.mean(axis=1).mean():.4f}')
print(f'anglez_z_all : {anglez_z_all.shape}')

## Step 3 – Data-Driven Window Length Selection ★

We scan m = 10 to 60 and compute a **separation score** = (max MP − min MP) / max MP  
for each candidate window length on a representative subject.  
Higher score → the matrix profile has a wider dynamic range → clearer motifs and discords.  
The best m is selected by argmax of this score — fully data-driven, no manual judgment.

In [ ]:
# Use first healthy subject for window selection
ref_for_window = np.where(y_train == 0)[0][0]
ts_ref = enmo_z_all[ref_for_window]

m_candidates = np.arange(10, 61, 5)
sep_scores   = []

for m_val in m_candidates:
    mp = stumpy.stump(ts_ref, m_val)
    mp_vals = mp[:, 0].astype(np.float64)
    sep = (np.max(mp_vals) - np.min(mp_vals)) / (np.max(mp_vals) + 1e-8)
    sep_scores.append(sep)

best_m_idx = np.argmax(sep_scores)
M = int(m_candidates[best_m_idx])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(m_candidates, sep_scores, 'o-', color='steelblue', lw=1.5)
ax.axvline(M, color='red', ls='--', label=f'Selected m={M} ({M*30//60}h)')
ax.set_xlabel('Window length m'); ax.set_ylabel('Separation score')
ax.set_title('Data-Driven Window Length Selection\n(max−min)/max of Matrix Profile')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Selected M = {M} steps = {M*30} min = {M*30/60:.1f} hours')
print(f'Separation score: {sep_scores[best_m_idx]:.4f}')

## Step 4 – Select Representative TS Systematically

Criteria: zero non-wear fraction, no low battery, highest enmo std (most active, informative).

In [ ]:
enmo_stds = X_train_raw[:, :, ENMO_IDX].std(axis=1)
candidate_mask = (
    (meta_train['nonwear_frac'] == 0) &
    (meta_train['has_low_battery'] == 0)
)
candidates = meta_train[candidate_mask].copy()
candidates['enmo_std'] = enmo_stds[candidates.index]
rep_idx = int(candidates['enmo_std'].idxmax())
rep_info = meta_train.loc[rep_idx]

print(f'Representative TS: index={rep_idx}, subject={rep_info["id"]}, sii={rep_info["sii_binary"]}')

ts_rep_enmo   = enmo_z_all[rep_idx]
ts_rep_anglez = anglez_z_all[rep_idx]
mp_rep        = stumpy.stump(ts_rep_enmo, M)

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(ts_rep_enmo,   color='steelblue', lw=0.9, label='enmo (z-scored)')
axes[0].set_title(f'Representative TS — subject {rep_info["id"]}, sii={rep_info["sii_binary"]}')
axes[0].set_ylabel('enmo'); axes[0].legend()
axes[1].plot(ts_rep_anglez, color='darkorange', lw=0.9, label='anglez (z-scored)')
axes[1].set_ylabel('anglez'); axes[1].set_xlabel('Time step (30-min)'); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 5 – Motif Discovery — Single TS

In [ ]:
motif_distances, motif_indices = stumpy.motifs(
    ts_rep_enmo, mp_rep[:, 0],
    max_motifs=3, cutoff=np.inf, min_neighbors=1
)

motif_colors = ['green', 'royalblue', 'purple']
print(f'Top-3 motifs (m={M}, {M*30//60}h):')
for i in range(len(motif_distances)):
    valid = [int(p) for p in motif_indices[i] if p >= 0]
    print(f'  Motif {i+1}: dist={float(motif_distances[i][0]):.4f}, positions={valid}')

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(ts_rep_enmo, color='black', lw=0.8, alpha=0.7)
handles = [mpatches.Patch(color='black', label='enmo (z-scored)')]
for i in range(len(motif_distances)):
    valid = [int(p) for p in motif_indices[i] if p >= 0]
    for pos in valid:
        axes[0].axvspan(pos, min(pos+M, 200), alpha=0.35, color=motif_colors[i])
    handles.append(mpatches.Patch(color=motif_colors[i], alpha=0.5,
                   label=f'Motif {i+1} (d={float(motif_distances[i][0]):.3f})'))
axes[0].legend(handles=handles, fontsize=9)
axes[0].set_title(f'Top-3 Motifs (m={M})')
axes[0].set_ylabel('enmo (z-scored)')

t_sub = np.arange(M)
for i in range(len(motif_distances)):
    valid = [int(p) for p in motif_indices[i] if p >= 0]
    for j, pos in enumerate(valid[:3]):
        seg = ts_rep_enmo[pos:pos+M]
        if len(seg) == M:
            axes[1].plot(t_sub, seg, color=motif_colors[i], lw=1.2,
                         alpha=0.9 if j==0 else 0.4,
                         label=f'Motif {i+1}' if j==0 else '')
axes[1].set_title('Motif Shape Overlay')
axes[1].set_xlabel('Relative step'); axes[1].set_ylabel('enmo (z-scored)')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## Step 6 – Discord Discovery — Single TS

In [ ]:
def get_discords(mp_dist, excl_zone, k=3):
    mp_copy = mp_dist.copy().astype(float)
    discords = []
    for _ in range(k):
        pos = int(np.argmax(mp_copy))
        discords.append((pos, float(mp_copy[pos])))
        mp_copy[max(0,pos-excl_zone):min(len(mp_copy),pos+excl_zone+1)] = -np.inf
    return discords

discords_rep = get_discords(mp_rep[:, 0], excl_zone=M//2, k=3)
discord_colors = ['red', 'orange', 'darkred']

print(f'Top-3 discords (m={M}):')
for i, (pos, dist) in enumerate(discords_rep):
    print(f'  Discord {i+1}: pos={pos}, dist={dist:.4f}')

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(ts_rep_enmo, color='black', lw=0.8, alpha=0.7)
handles = [mpatches.Patch(color='black', label='enmo (z-scored)')]
for i, (pos, dist) in enumerate(discords_rep):
    axes[0].axvspan(pos, min(pos+M,200), alpha=0.4, color=discord_colors[i])
    handles.append(mpatches.Patch(color=discord_colors[i], alpha=0.6,
                   label=f'Discord {i+1} @ {pos} (d={dist:.3f})'))
axes[0].legend(handles=handles, fontsize=9)
axes[0].set_title(f'Top-3 Discords (m={M})')
axes[0].set_ylabel('enmo (z-scored)')

axes[1].plot(mp_rep[:, 0], color='gray', lw=0.9, label='Matrix Profile')
for i, (pos, dist) in enumerate(discords_rep):
    axes[1].scatter(pos, dist, color=discord_colors[i], s=80, zorder=5, label=f'Discord {i+1}')
axes[1].set_title('Matrix Profile — Discord Peaks')
axes[1].set_xlabel('Subsequence start'); axes[1].set_ylabel('Distance')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## Step 7 – Motif vs Discord Comparison

In [ ]:
top_motif_pos   = int(motif_indices[0][0])
top_discord_pos = discords_rep[0][0]
t_sub = np.arange(M)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(ts_rep_enmo, color='black', lw=0.8, alpha=0.7)
axes[0].axvspan(top_motif_pos,   min(top_motif_pos+M,200),   alpha=0.4, color='green', label='Top motif')
axes[0].axvspan(top_discord_pos, min(top_discord_pos+M,200), alpha=0.4, color='red',   label='Top discord')
axes[0].set_title('Motif vs Discord on Same TS'); axes[0].set_ylabel('enmo'); axes[0].legend()

motif_seg   = ts_rep_enmo[top_motif_pos:top_motif_pos+M]
discord_seg = ts_rep_enmo[top_discord_pos:top_discord_pos+M]
axes[1].plot(t_sub, motif_seg,   color='green', lw=2, label=f'Top motif (pos={top_motif_pos})')
axes[1].plot(t_sub, discord_seg, color='red',   lw=2, label=f'Top discord (pos={top_discord_pos})')
axes[1].set_title('Shape Comparison'); axes[1].set_xlabel('Relative step')
axes[1].set_ylabel('enmo (z-scored)'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Motif distance   (repetitive): {float(mp_rep[top_motif_pos,0]):.4f}')
print(f'Discord distance (anomalous) : {discords_rep[0][1]:.4f}')

## Step 8 – Global Per-Subject Matrix Profile

Computed **per subject separately** — no concatenation — to avoid artificial boundary effects.

In [ ]:
global_results = []
print(f'Computing matrix profiles for {N} training TS (m={M})...')

for i in range(N):
    arr_e = enmo_z_all[i]
    arr_a = anglez_z_all[i]
    sii   = int(y_train[i])

    mp_e = stumpy.stump(arr_e, M)
    mp_a = stumpy.stump(arr_a, M)

    mp_e_dist = np.where(np.isinf(mp_e[:,0]), np.nan, mp_e[:,0].astype(float))
    mp_a_dist = np.where(np.isinf(mp_a[:,0]), np.nan, mp_a[:,0].astype(float))

    motif_pos_e  = int(np.nanargmin(mp_e_dist))
    discord_e    = get_discords(mp_e[:,0], excl_zone=M//2, k=1)[0]
    motif_pos_a  = int(np.nanargmin(mp_a_dist))
    discord_a    = get_discords(mp_a[:,0], excl_zone=M//2, k=1)[0]

    global_results.append({
        'ts_idx'            : i,
        'subject_id'        : meta_train.loc[i, 'id'] if i < len(meta_train) else -1,
        'sii'               : sii,
        'enmo_motif_pos'    : motif_pos_e,
        'enmo_motif_dist'   : float(mp_e_dist[motif_pos_e]),
        'enmo_discord_pos'  : discord_e[0],
        'enmo_discord_dist' : discord_e[1],
        'enmo_regularity'   : float(np.nanmin(mp_e_dist)),
        'anglez_motif_pos'  : motif_pos_a,
        'anglez_motif_dist' : float(mp_a_dist[motif_pos_a]),
        'anglez_discord_pos': discord_a[0],
        'anglez_discord_dist':discord_a[1],
        'anglez_regularity' : float(np.nanmin(mp_a_dist)),
    })
    if (i+1) % 500 == 0:
        print(f'  {i+1}/{N} done...')

global_df = pd.DataFrame(global_results)
print(f'Done. Shape: {global_df.shape}')
print(global_df.groupby('sii')[['enmo_regularity','enmo_discord_dist','anglez_regularity']].mean().round(4))

## Step 9 – Consensus Motif by Class ± Std Shading ★

In [ ]:
# Collect motif shapes per class
consensus = {0: [], 1: []}
for _, row in global_df.iterrows():
    pos = int(row['enmo_motif_pos'])
    idx = int(row['ts_idx'])
    sii = int(row['sii'])
    seg = enmo_z_all[idx, pos:pos+M]
    if len(seg) == M:
        consensus[sii].append(seg)

shapes0 = np.array(consensus[0])
shapes1 = np.array(consensus[1])
mean0, std0 = shapes0.mean(axis=0), shapes0.std(axis=0)
mean1, std1 = shapes1.mean(axis=0), shapes1.std(axis=0)
l2_dist = euclidean(mean0, mean1)
t_sub   = np.arange(M)

print(f'Class 0 motifs: {len(shapes0)}, Class 1 motifs: {len(shapes1)}')
print(f'L2 distance between consensus motifs: {l2_dist:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class 0
for seg in shapes0[::15]:
    axes[0].plot(t_sub, seg, color='steelblue', lw=0.3, alpha=0.3)
axes[0].plot(t_sub, mean0, color='steelblue', lw=2.5, label='Consensus')
axes[0].fill_between(t_sub, mean0-std0, mean0+std0, alpha=0.2, color='steelblue', label='±1 std')
axes[0].set_title('Class 0 (Non-problematic)')
axes[0].set_xlabel('Relative step'); axes[0].set_ylabel('enmo (z-scored)'); axes[0].legend()

# Class 1
for seg in shapes1[::15]:
    axes[1].plot(t_sub, seg, color='tomato', lw=0.3, alpha=0.3)
axes[1].plot(t_sub, mean1, color='tomato', lw=2.5, label='Consensus')
axes[1].fill_between(t_sub, mean1-std1, mean1+std1, alpha=0.2, color='tomato', label='±1 std')
axes[1].set_title('Class 1 (Problematic)')
axes[1].set_xlabel('Relative step'); axes[1].legend()

# Overlay
axes[2].plot(t_sub, mean0, color='steelblue', lw=2, label='Class 0')
axes[2].plot(t_sub, mean1, color='tomato',    lw=2, label='Class 1')
axes[2].fill_between(t_sub, mean0-std0, mean0+std0, alpha=0.1, color='steelblue')
axes[2].fill_between(t_sub, mean1-std1, mean1+std1, alpha=0.1, color='tomato')
axes[2].fill_between(t_sub, mean0, mean1, alpha=0.15, color='gray')
axes[2].set_title(f'Overlay (L2={l2_dist:.3f})')
axes[2].set_xlabel('Relative step'); axes[2].legend()

plt.suptitle('Consensus Motif per Class ± 1 std (enmo)', fontsize=13)
plt.tight_layout(); plt.show()

## Step 10 – Motif Regularity Score by Class

In [ ]:
reg0 = global_df[global_df['sii']==0]['enmo_regularity'].values
reg1 = global_df[global_df['sii']==1]['enmo_regularity'].values
stat_r, pval_r = sp_stats.mannwhitneyu(reg0, reg1, alternative='two-sided')

print('=== Motif Regularity Score (MP minimum) ===')
print(f'Class 0: mean={reg0.mean():.4f}  median={np.median(reg0):.4f}  std={reg0.std():.4f}')
print(f'Class 1: mean={reg1.mean():.4f}  median={np.median(reg1):.4f}  std={reg1.std():.4f}')
print(f'Mann-Whitney U: p={pval_r:.4f} → {"significant" if pval_r<0.05 else "not significant"}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([reg0, reg1], labels=['Class 0','Class 1'], patch_artist=True,
                boxprops=dict(facecolor='steelblue',alpha=0.5),
                medianprops=dict(color='black',lw=2))
axes[0].set_ylabel('Regularity (MP min)'); axes[0].set_title(f'Regularity by Class (p={pval_r:.4f})')

axes[1].hist(reg0, bins=30, alpha=0.6, color='steelblue', density=True, label='Class 0', edgecolor='white')
axes[1].hist(reg1, bins=30, alpha=0.6, color='tomato',    density=True, label='Class 1', edgecolor='white')
axes[1].axvline(reg0.mean(), color='steelblue', ls='--', lw=1.5)
axes[1].axvline(reg1.mean(), color='tomato',    ls='--', lw=1.5)
axes[1].set_xlabel('Regularity score'); axes[1].set_title('Distribution'); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 11 – Class-Level Discord Analysis + Mann-Whitney

In [ ]:
disc0 = global_df[global_df['sii']==0]['enmo_discord_dist'].values
disc1 = global_df[global_df['sii']==1]['enmo_discord_dist'].values
stat_d, pval_d = sp_stats.mannwhitneyu(disc0, disc1, alternative='two-sided')

print('=== Discord Distance by Class ===')
print(f'Class 0: mean={disc0.mean():.4f}  median={np.median(disc0):.4f}')
print(f'Class 1: mean={disc1.mean():.4f}  median={np.median(disc1):.4f}')
print(f'Mann-Whitney U: p={pval_d:.4f} → {"significant" if pval_d<0.05 else "not significant"}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([disc0, disc1], labels=['Class 0','Class 1'], patch_artist=True,
                boxprops=dict(facecolor='tomato',alpha=0.5),
                medianprops=dict(color='black',lw=2))
axes[0].set_ylabel('Discord distance'); axes[0].set_title(f'Discord Extremeness (p={pval_d:.4f})')

axes[1].hist(disc0, bins=30, alpha=0.6, color='steelblue', density=True, label='Class 0', edgecolor='white')
axes[1].hist(disc1, bins=30, alpha=0.6, color='tomato',    density=True, label='Class 1', edgecolor='white')
axes[1].set_xlabel('Discord distance'); axes[1].set_title('Distribution'); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 12 – Fisher Exact Test: Discord Enrichment ★

Are the top 10% most anomalous subjects disproportionately sii=1?  
Fisher exact test on a 2×2 contingency table — more direct than comparing means.

In [ ]:
disc_all = global_df['enmo_discord_dist'].values
lbl_all  = global_df['sii'].values
threshold_val = np.percentile(disc_all, 90)
extreme_mask  = disc_all >= threshold_val

e_sii1 = np.sum(lbl_all[extreme_mask]  == 1)
e_sii0 = np.sum(lbl_all[extreme_mask]  == 0)
n_sii1 = np.sum(lbl_all[~extreme_mask] == 1)
n_sii0 = np.sum(lbl_all[~extreme_mask] == 0)

odds_ratio, pval_fisher = fisher_exact(
    [[e_sii1, e_sii0], [n_sii1, n_sii0]], alternative='greater'
)

print(f'=== Discord Enrichment Test (top 10% most anomalous) ===')
print(f'Threshold MP value      : {threshold_val:.4f}')
print(f'Extreme discord TS      : {extreme_mask.sum()}')
print(f'  sii=1 (problematic)   : {e_sii1} ({100*e_sii1/extreme_mask.sum():.1f}%)')
print(f'  sii=0 (healthy)       : {e_sii0} ({100*e_sii0/extreme_mask.sum():.1f}%)')
print(f'Background              : {(~extreme_mask).sum()}')
print(f'  sii=1                 : {n_sii1} ({100*n_sii1/(~extreme_mask).sum():.1f}%)')
print(f'  sii=0                 : {n_sii0} ({100*n_sii0/(~extreme_mask).sum():.1f}%)')
print(f'Odds ratio              : {odds_ratio:.4f}')
print(f'Fisher exact p (1-sided): {pval_fisher:.4f}')
print(f'→ {"Significant enrichment of sii=1 in extreme discords ✓" if pval_fisher<0.05 else "No significant enrichment"}')

# Stacked bar
fig, ax = plt.subplots(figsize=(6, 4))
groups = ['Top 10%\n(extreme)', 'Bottom 90%\n(normal)']
sii1_pct = [100*e_sii1/extreme_mask.sum(), 100*n_sii1/(~extreme_mask).sum()]
sii0_pct = [100*e_sii0/extreme_mask.sum(), 100*n_sii0/(~extreme_mask).sum()]
ax.bar(groups, sii0_pct, label='sii=0', color='steelblue', alpha=0.8)
ax.bar(groups, sii1_pct, bottom=sii0_pct, label='sii=1', color='tomato', alpha=0.8)
ax.set_ylabel('Percentage (%)'); ax.set_title(f'Discord Enrichment (Fisher p={pval_fisher:.4f})')
ax.legend(); ax.set_ylim(0, 110)
plt.tight_layout(); plt.show()

## Step 13 – MP Distribution by Class ★

Full distribution of MP values per class — not just means.  
Lower MP → more repetitive behavior (shapelet candidate).  
Higher MP → more anomalous behavior.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(disc0, bins=40, alpha=0.6, density=True, color='steelblue', label='Class 0', edgecolor='white')
axes[0].hist(disc1, bins=40, alpha=0.6, density=True, color='tomato',    label='Class 1', edgecolor='white')
axes[0].axvline(disc0.mean(), color='steelblue', ls='--', lw=1.5, label=f'Mean cls0={disc0.mean():.3f}')
axes[0].axvline(disc1.mean(), color='tomato',    ls='--', lw=1.5, label=f'Mean cls1={disc1.mean():.3f}')
axes[0].set_xlabel('Max MP distance (discord score)')
axes[0].set_ylabel('Density')
axes[0].set_title('MP Discord Distribution by Class\n(higher = more anomalous)')
axes[0].legend(fontsize=8)

axes[1].hist(reg0, bins=40, alpha=0.6, density=True, color='steelblue', label='Class 0', edgecolor='white')
axes[1].hist(reg1, bins=40, alpha=0.6, density=True, color='tomato',    label='Class 1', edgecolor='white')
axes[1].axvline(reg0.mean(), color='steelblue', ls='--', lw=1.5, label=f'Mean cls0={reg0.mean():.3f}')
axes[1].axvline(reg1.mean(), color='tomato',    ls='--', lw=1.5, label=f'Mean cls1={reg1.mean():.3f}')
axes[1].set_xlabel('Min MP distance (regularity score)')
axes[1].set_ylabel('Density')
axes[1].set_title('MP Regularity Distribution by Class\n(lower = more regular behavior)')
axes[1].legend(fontsize=8)

plt.suptitle('Full MP Distribution by Class', fontsize=13)
plt.tight_layout(); plt.show()

print('Interpretation:')
print('Low MP values → motif candidates (recurring patterns → shapelet candidates)')
print('High MP values → discord candidates (anomalous, rare patterns)')

## Step 14 – Discord Density Over Time (Early / Mid / Late)

In [ ]:
def classify_region(pos, total=200):
    if pos < total//3: return 'early'
    elif pos < 2*total//3: return 'middle'
    else: return 'late'

global_df['discord_region'] = global_df['enmo_discord_pos'].apply(classify_region)
region_counts = global_df.groupby(['sii','discord_region']).size().unstack(fill_value=0)
region_pct    = region_counts.div(region_counts.sum(axis=1), axis=0) * 100

print('Discord region distribution (%):')
print(region_pct.round(1))

fig, ax = plt.subplots(figsize=(7, 4))
region_pct.T.plot(kind='bar', ax=ax, color=['steelblue','tomato'],
                  edgecolor='white', width=0.6)
ax.set_xlabel('Recording region'); ax.set_ylabel('% of discords')
ax.set_title('Discord Location by Region and Class')
ax.set_xticklabels(['Early (0–66)','Late (134–200)','Middle (67–133)'], rotation=0)
ax.legend(['Class 0','Class 1'])
plt.tight_layout(); plt.show()

## Step 15 – Cross-Series Motif + Discord Matching

In [ ]:
# Reference motif
ref_motif_pos = int(motif_indices[0][0])
ref_motif_seg = ts_rep_enmo[ref_motif_pos:ref_motif_pos+M]

cross_matches = []
for i in range(N):
    if i == rep_idx: continue
    target = enmo_z_all[i]
    matches = stumpy.match(ref_motif_seg, target, max_distance=np.inf)
    if len(matches) > 0:
        cross_matches.append({'ts_idx':i, 'sii':int(y_train[i]),
                              'match_pos':int(matches[0,1]), 'match_dist':float(matches[0,0])})

cross_df = pd.DataFrame(cross_matches).sort_values('match_dist')
print(f'Top-5 cross-series motif matches:')
print(cross_df.head(5)[['ts_idx','sii','match_pos','match_dist']].to_string(index=False))

# Reference discord
ref_discord_pos = discords_rep[0][0]
ref_discord_seg = ts_rep_enmo[ref_discord_pos:ref_discord_pos+M]

cross_disc = []
for i in range(N):
    if i == rep_idx: continue
    target  = enmo_z_all[i]
    matches = stumpy.match(ref_discord_seg, target, max_distance=np.inf)
    if len(matches) > 0:
        cross_disc.append({'ts_idx':i, 'sii':int(y_train[i]),
                           'match_pos':int(matches[0,1]), 'match_dist':float(matches[0,0])})

cross_disc_df = pd.DataFrame(cross_disc).sort_values('match_dist')
print(f'\nTop-5 cross-series discord matches:')
print(cross_disc_df.head(5)[['ts_idx','sii','match_pos','match_dist']].to_string(index=False))

# Visualize top motif match
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(ts_rep_enmo, color='gray', lw=0.8, alpha=0.7)
axes[0].axvspan(ref_motif_pos, ref_motif_pos+M, alpha=0.4, color='green')
axes[0].set_title(f'Reference motif (rep TS, pos={ref_motif_pos})')
axes[0].set_ylabel('enmo (z-scored)')
best_match = cross_df.iloc[0]
target_ts  = enmo_z_all[int(best_match['ts_idx'])]
mpos       = int(best_match['match_pos'])
axes[1].plot(target_ts, color='gray', lw=0.8, alpha=0.7)
axes[1].axvspan(mpos, mpos+M, alpha=0.4, color='royalblue')
axes[1].set_title(f'Best match: TS {best_match["ts_idx"]}, sii={best_match["sii"]}, dist={best_match["match_dist"]:.4f}')
axes[1].set_xlabel('Time step'); axes[1].set_ylabel('enmo (z-scored)')
plt.tight_layout(); plt.show()

## Step 16 – Shapelet Bridge: Consensus Motif as Primitive Shapelet ★

We use the class-0 consensus motif as a primitive shapelet and compute  
the minimum sliding distance from every TS to this pattern.  
This directly addresses the guidelines' requirement to discuss the  
relationship between motifs and shapelets — and tests whether  
a single consensus pattern can discriminate classes.

In [ ]:
shapelet = mean0  # class-0 consensus motif

print('Computing sliding distance from each TS to class-0 consensus motif...')
min_dists = []
for i in range(N):
    ts_i = enmo_z_all[i]
    best = min(
        np.linalg.norm(ts_i[j:j+M] - shapelet)
        for j in range(len(ts_i)-M+1)
    )
    min_dists.append(best)

min_dists = np.array(min_dists)

# Threshold classifier: median distance of class-0 TS
threshold_shap = np.median(min_dists[y_train==0])
pred = (min_dists > threshold_shap).astype(int)
acc  = (pred == y_train).mean()

print(f'\nShapelet classification (threshold = median dist of class 0):')
print(f'  Threshold : {threshold_shap:.4f}')
print(f'  Accuracy  : {acc:.3f}  (random baseline ~0.5)')
print(f'\nInterpretation:')
if acc < 0.55:
    print('  Accuracy near chance → single consensus motif does not discriminate classes.')
    print('  This is scientifically meaningful: problematic internet use is not captured')
    print('  by a single recurring activity pattern. Richer shapelet sets are needed.')
else:
    print('  Above-chance accuracy → consensus motif carries some discriminative information.')

# Plot distance distributions
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(min_dists[y_train==0], bins=30, alpha=0.6, color='steelblue',
        density=True, label='Class 0', edgecolor='white')
ax.hist(min_dists[y_train==1], bins=30, alpha=0.6, color='tomato',
        density=True, label='Class 1', edgecolor='white')
ax.axvline(threshold_shap, color='black', ls='--', lw=1.5,
           label=f'Threshold={threshold_shap:.3f}')
ax.set_xlabel('Min distance to class-0 consensus motif')
ax.set_ylabel('Density')
ax.set_title(f'Shapelet Distance Distribution (acc={acc:.3f})')
ax.legend()
plt.tight_layout(); plt.show()

## Step 17 – Joint enmo + anglez Discord ★

Timesteps where both enmo and anglez are simultaneously anomalous —  
multichannel anomaly events more likely to represent genuine behavioral instability.

In [ ]:
def find_joint_discords(mp_e_dist, mp_a_dist, excl_zone, k=10, tolerance=12):
    d_enmo   = [pos for pos,_ in get_discords(mp_e_dist, excl_zone, k=k)]
    d_anglez = [pos for pos,_ in get_discords(mp_a_dist, excl_zone, k=k)]
    joint = []
    for de in d_enmo:
        for da in d_anglez:
            if abs(de-da) <= tolerance:
                joint.append({'enmo_pos':de,'anglez_pos':da,
                              'offset':abs(de-da),
                              'enmo_dist':float(mp_e_dist[de]),
                              'anglez_dist':float(mp_a_dist[da])})
    return pd.DataFrame(joint).drop_duplicates('enmo_pos').sort_values('enmo_dist',ascending=False) \
           if joint else pd.DataFrame()

# On representative TS
mp_rep_a = stumpy.stump(ts_rep_anglez, M)
joint_rep = find_joint_discords(mp_rep[:,0], mp_rep_a[:,0], excl_zone=M//2)
print(f'Joint discords on representative TS: {len(joint_rep)}')
if len(joint_rep): print(joint_rep.head().to_string(index=False))

# Global joint discord counts
print('\nComputing joint discords globally...')
joint_counts = []
for i in range(N):
    mp_e = stumpy.stump(enmo_z_all[i],   M)
    mp_a = stumpy.stump(anglez_z_all[i], M)
    j = find_joint_discords(mp_e[:,0], mp_a[:,0], excl_zone=M//2)
    joint_counts.append({'ts_idx':i,'sii':int(y_train[i]),'joint_count':len(j)})

jc_df  = pd.DataFrame(joint_counts)
jc0    = jc_df[jc_df['sii']==0]['joint_count'].values
jc1    = jc_df[jc_df['sii']==1]['joint_count'].values
stat_j, pval_j = sp_stats.mannwhitneyu(jc0, jc1, alternative='two-sided')

print(f'Joint discord count — Class 0: mean={jc0.mean():.3f}, Class 1: mean={jc1.mean():.3f}')
print(f'Mann-Whitney p={pval_j:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if len(joint_rep):
    axes[0].plot(enmo_z_all[rep_idx],   color='steelblue', lw=0.9, alpha=0.8, label='enmo')
    axes[0].plot(anglez_z_all[rep_idx], color='darkorange', lw=0.9, alpha=0.8, label='anglez')
    for _, jrow in joint_rep.head(3).iterrows():
        axes[0].axvspan(int(jrow['enmo_pos']), min(int(jrow['enmo_pos'])+M,200),
                        alpha=0.3, color='purple')
    axes[0].plot([],[],color='purple',alpha=0.5,lw=8,label='Joint discord')
    axes[0].set_title('Joint enmo+anglez Discords (rep TS)')
    axes[0].set_xlabel('Time step'); axes[0].set_ylabel('z-scored'); axes[0].legend(fontsize=8)

axes[1].boxplot([jc0, jc1], labels=['Class 0','Class 1'], patch_artist=True,
                boxprops=dict(facecolor='mediumpurple',alpha=0.5),
                medianprops=dict(color='black',lw=2))
axes[1].set_ylabel('Joint discord count'); axes[1].set_title(f'Multichannel Anomalies (p={pval_j:.4f})')
plt.tight_layout(); plt.show()

## Step 18 – Repeat Key Analysis for anglez

In [ ]:
# anglez consensus motif
consensus_a = {0: [], 1: []}
for _, row in global_df.iterrows():
    pos = int(row['anglez_motif_pos'])
    idx = int(row['ts_idx'])
    sii = int(row['sii'])
    seg = anglez_z_all[idx, pos:pos+M]
    if len(seg) == M:
        consensus_a[sii].append(seg)

mean_a0, std_a0 = np.array(consensus_a[0]).mean(axis=0), np.array(consensus_a[0]).std(axis=0)
mean_a1, std_a1 = np.array(consensus_a[1]).mean(axis=0), np.array(consensus_a[1]).std(axis=0)
l2_anglez = euclidean(mean_a0, mean_a1)

# anglez regularity
areg0 = global_df[global_df['sii']==0]['anglez_regularity'].values
areg1 = global_df[global_df['sii']==1]['anglez_regularity'].values
_, pval_ar = sp_stats.mannwhitneyu(areg0, areg1, alternative='two-sided')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
t_sub = np.arange(M)

axes[0].plot(t_sub, mean_a0, color='steelblue', lw=2, label='Class 0')
axes[0].fill_between(t_sub, mean_a0-std_a0, mean_a0+std_a0, alpha=0.2, color='steelblue')
axes[0].plot(t_sub, mean_a1, color='tomato',    lw=2, label='Class 1')
axes[0].fill_between(t_sub, mean_a1-std_a1, mean_a1+std_a1, alpha=0.2, color='tomato')
axes[0].set_title(f'anglez Consensus Motif (L2={l2_anglez:.3f})')
axes[0].set_xlabel('Relative step'); axes[0].set_ylabel('anglez (z-scored)'); axes[0].legend()

# enmo vs anglez consensus overlay
axes[1].plot(t_sub, mean0,   color='steelblue', lw=2,  ls='-',  label='enmo cls0')
axes[1].plot(t_sub, mean1,   color='tomato',    lw=2,  ls='-',  label='enmo cls1')
axes[1].plot(t_sub, mean_a0, color='steelblue', lw=2,  ls='--', label='anglez cls0')
axes[1].plot(t_sub, mean_a1, color='tomato',    lw=2,  ls='--', label='anglez cls1')
axes[1].set_title('enmo vs anglez Consensus Motif'); axes[1].set_xlabel('Relative step')
axes[1].legend(fontsize=8)

# Regularity comparison
axes[2].boxplot([areg0, areg1], labels=['Class 0','Class 1'], patch_artist=True,
                boxprops=dict(facecolor='darkorange',alpha=0.5),
                medianprops=dict(color='black',lw=2))
axes[2].set_ylabel('anglez regularity (MP min)')
axes[2].set_title(f'anglez Regularity by Class (p={pval_ar:.4f})')

plt.suptitle('anglez Analysis', fontsize=13)
plt.tight_layout(); plt.show()

print('Summary comparison:')
print(f'  enmo   L2 consensus distance: {l2_dist:.4f}')
print(f'  anglez L2 consensus distance: {l2_anglez:.4f}')
print(global_df.groupby('sii')[['enmo_regularity','anglez_regularity']].mean().round(4))

## Step 19 – Shapelet Alignment Placeholder

Full shapelet alignment (DTW comparison of extracted shapelets with motifs/discords)  
is deferred to after the classification notebook where shapelets are extracted.

**Plan**: Load top-K shapelets from `ShapeletTransformClassifier`, compute DTW to all motif  
and discord positions from `global_df`, report match table with threshold 0.2.

In [ ]:
print('Shapelet alignment: deferred to after classification notebook.')
print('global_df is available — contains all motif/discord positions for every training TS.')

## Summary

| Step | What | From teammate | Ours |
|------|------|--------------|------|
| 3 | Window selection | Separation score ★ | Added |
| 9 | Consensus motif ± std | ±std shading ★ | Added |
| 12 | Fisher exact test | Discord enrichment ★ | Added |
| 13 | MP distribution by class | Full distribution ★ | Added |
| 16 | Shapelet bridge | Consensus as shapelet ★ | Added |
| 8 | Per-subject MP | No concatenation | Ours |
| 14 | Discord density | Early/mid/late | Ours |
| 15 | Cross-series matching | Explicit distances | Ours |
| 17 | Joint discord | enmo+anglez | Ours |
| 18 | anglez analysis | Second channel | Ours |

**Data used**: `X_train_raw.npy`, `y_train.npy`, `meta_train.csv` — Module 0 outputs, no re-cleaning.